# CNN sur Fashion-MNIST
Même démarche que pour le MLP : on calibre d'abord (largeur × epochs, sans dropout vs dropout classique) pour trouver un régime de sous-apprentissage, puis on lance la grille des durées d'early dropout. À lancer sur Colab avec le GPU.

In [4]:
# Colab : monte le Drive et se place dans le repo
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
!git pull

Mounted at /content/drive
/content/drive/MyDrive/early-dropout
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 7), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.72 KiB | 7.00 KiB/s, done.
From https://github.com/Felix-prjcts/early-dropout-study
   0813ca2..e0efe53  master     -> origin/master
Updating 0813ca2..e0efe53
Fast-forward
 __pycache__/train.cpython-311.pyc | Bin 4515 -> 4647 bytes
 train.py                          |  14 ++++++++++----
 2 files changed, 10 insertions(+), 4 deletions(-)


In [ ]:
import os, glob, time, importlib
import pandas as pd
import torch
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)   # prend en compte un git pull récent
from config import BATCH, P, EARLY_GRID
from models import cnn
from train import entrainer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr = tr.data.float().unsqueeze(1) / 255      # (N, 1, 28, 28) : on garde la forme d'image pour le CNN
Xte = te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split de validation que pour le MLP
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]

# normalisation avec les stats du train uniquement
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

data = dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte)
data = {k: v.to(device) for k, v in data.items()}       # tout sur le GPU
print(Xtr.shape, Xva.shape, Xte.shape)

torch.Size([50000, 1, 28, 28]) torch.Size([10000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


In [ ]:
CADRE = "cnn_fmnist"
OUT = "resultats"

# combien de temps prend une epoch ? (pour dimensionner les grilles)
torch.manual_seed(0)
model, opt = cnn(1, 10, largeur=8)
model.to(device)
t0 = time.time()
entrainer(model, opt, data, epochs=1, t_early=0, p=P, batch=BATCH)
print(f"{time.time() - t0:.1f} s par epoch")

epoch  0 | rate 0.000 | train loss 0.705 acc 0.755 | val loss 0.733 acc 0.743
5.0 s par epoch


## Calibration : trouver un régime de sous-apprentissage
On compare sans dropout et dropout classique. delta = acc val (dropout classique) - acc val (sans dropout) : si delta < 0 le modèle sous-apprend, c'est ce qu'on cherche.

In [ ]:
CALIB_LARGEURS = [4, 8, 16]
CALIB_EPOCHS = [10, 20]
CALIB_GRID = [0, 1]          # 0 = sans dropout, 1 = dropout classique
CALIB_SEEDS = [0, 1]

for L in CALIB_LARGEURS:
    for E in CALIB_EPOCHS:
        for frac in CALIB_GRID:
            for seed in CALIB_SEEDS:
                fichier = f"{OUT}/{CADRE}_w{L}_e{E}_frac{frac}_s{seed}.csv"
                if os.path.exists(fichier):
                    continue
                torch.manual_seed(seed)
                model, opt = cnn(1, 10, largeur=L)
                model.to(device)
                hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=P, batch=BATCH)
                hist.assign(cadre=CADRE, width=L, epochs=E, frac=frac, seed=seed).to_csv(fichier, index=False)

epoch  0 | rate 0.000 | train loss 0.797 acc 0.718 | val loss 0.820 acc 0.709
epoch  1 | rate 0.000 | train loss 0.692 acc 0.742 | val loss 0.727 acc 0.728
epoch  2 | rate 0.000 | train loss 0.611 acc 0.780 | val loss 0.640 acc 0.768
epoch  3 | rate 0.000 | train loss 0.576 acc 0.793 | val loss 0.602 acc 0.781
epoch  4 | rate 0.000 | train loss 0.546 acc 0.798 | val loss 0.575 acc 0.791
epoch  5 | rate 0.000 | train loss 0.519 acc 0.808 | val loss 0.549 acc 0.798
epoch  6 | rate 0.000 | train loss 0.516 acc 0.813 | val loss 0.543 acc 0.807
epoch  7 | rate 0.000 | train loss 0.505 acc 0.814 | val loss 0.534 acc 0.802
epoch  8 | rate 0.000 | train loss 0.495 acc 0.817 | val loss 0.525 acc 0.805
epoch  9 | rate 0.000 | train loss 0.493 acc 0.819 | val loss 0.524 acc 0.807
epoch  0 | rate 0.000 | train loss 0.796 acc 0.719 | val loss 0.816 acc 0.711
epoch  1 | rate 0.000 | train loss 0.667 acc 0.768 | val loss 0.688 acc 0.759
epoch  2 | rate 0.000 | train loss 0.617 acc 0.783 | val loss 0.

In [ ]:
cal = pd.concat(pd.read_csv(f) for f in glob.glob(f"{OUT}/{CADRE}_w*_e*_frac*_s*.csv"))
cal = cal[cal.width.isin(CALIB_LARGEURS) & cal.epochs.isin(CALIB_EPOCHS) & cal.frac.isin(CALIB_GRID)]

# valeur finale = moyenne des 5 dernières epochs de chaque run
fin = (cal[cal.epoch >= cal.epochs - 5]
       .groupby(["width", "epochs", "frac", "seed"])[["train_loss", "val_loss", "val_acc"]].mean()
       .reset_index())

t = fin.pivot_table(index=["width", "epochs", "seed"], columns="frac", values="val_acc")
t["delta"] = (t[1] - t[0]) * 100                         # dropout classique - sans dropout, en points
tab = t.groupby(["width", "epochs"]).agg(
    acc_sans_dropout=(0, "mean"),
    delta_moy=("delta", "mean"),
    delta_min=("delta", "min"),
    delta_max=("delta", "max"),
)
tab["sous_apprentissage"] = tab.delta_max <= -0.5         # toutes les seeds perdent au moins 0.5 pt
tab.round(3)

acc_sans_dropout  delta_moy  delta_min  delta_max  \
width epochs                                                      
4     10                 0.804     -1.436     -2.030     -0.842   
      20                 0.840     -0.645     -1.360      0.070   
8     10                 0.860     -0.966     -1.566     -0.366   
      20                 0.891     -0.737     -0.960     -0.514   
16    10                 0.887     -0.003     -0.278      0.272   
      20                 0.911      0.021     -0.190      0.232   

              sous_apprentissage  
width epochs                      
4     10                    True  
      20                   False  
8     10                   False  
      20                    True  
16    10                   False  
      20                   False

On retient une case où `sous_apprentissage` est True, avec une accuracy correcte. Ensuite on lance la grille complète comme pour le MLP.

In [ ]:
LARGEUR = 8      # à remplacer par la largeur retenue
EPOCHS = 20      # à remplacer par le budget retenu
SEEDS = [0, 1, 2, 3, 4]

for frac in EARLY_GRID:
    for seed in SEEDS:
        fichier = f"{OUT}/{CADRE}_w{LARGEUR}_e{EPOCHS}_frac{frac}_s{seed}.csv"
        if os.path.exists(fichier):
            continue
        torch.manual_seed(seed)
        model, opt = cnn(1, 10, largeur=LARGEUR)
        model.to(device)
        hist = entrainer(model, opt, data, EPOCHS, t_early=round(frac * EPOCHS), p=P, batch=BATCH)
        hist.assign(cadre=CADRE, width=LARGEUR, epochs=EPOCHS, frac=frac, seed=seed).to_csv(fichier, index=False)

epoch  0 | rate 0.000 | train loss 0.710 acc 0.731 | val loss 0.737 acc 0.725
epoch  1 | rate 0.000 | train loss 0.586 acc 0.776 | val loss 0.610 acc 0.765
epoch  2 | rate 0.000 | train loss 0.552 acc 0.798 | val loss 0.576 acc 0.791
epoch  3 | rate 0.000 | train loss 0.475 acc 0.822 | val loss 0.504 acc 0.811
epoch  4 | rate 0.000 | train loss 0.432 acc 0.839 | val loss 0.466 acc 0.828
epoch  5 | rate 0.000 | train loss 0.417 acc 0.846 | val loss 0.445 acc 0.839
epoch  6 | rate 0.000 | train loss 0.384 acc 0.863 | val loss 0.416 acc 0.851
epoch  7 | rate 0.000 | train loss 0.375 acc 0.859 | val loss 0.411 acc 0.849
epoch  8 | rate 0.000 | train loss 0.333 acc 0.879 | val loss 0.367 acc 0.870
epoch  9 | rate 0.000 | train loss 0.326 acc 0.882 | val loss 0.358 acc 0.872
epoch 10 | rate 0.000 | train loss 0.318 acc 0.886 | val loss 0.358 acc 0.873
epoch 11 | rate 0.000 | train loss 0.303 acc 0.891 | val loss 0.342 acc 0.880
epoch 12 | rate 0.000 | train loss 0.297 acc 0.892 | val loss 0.

In [ ]:
# gain par rapport à "sans dropout", seed par seed, avec IC à 95 % (Student, 5 seeds)
metriques = ["train_loss", "val_loss", "val_acc", "test_acc"]
fin = {}
for frac in EARLY_GRID:
    fin[frac] = pd.DataFrame(
        [pd.read_csv(f"{OUT}/{CADRE}_w{LARGEUR}_e{EPOCHS}_frac{frac}_s{s}.csv").tail(5)[metriques].mean() for s in SEEDS],
        index=SEEDS)

res = {}
for frac in EARLY_GRID:
    if frac == 0:
        continue
    d = fin[frac] - fin[0]
    d[["val_acc", "test_acc"]] *= 100                     # accuracy en points
    ic = 2.776 * d.std() / len(SEEDS) ** 0.5
    res[frac] = pd.concat([d.mean(), ic.add_prefix("ic_")])
pd.DataFrame(res).T.round(3)

,train_loss,val_loss,val_acc,test_acc,ic_train_loss,ic_val_loss,ic_val_acc,ic_test_acc
0.1,-0.003,-0.002,-0.026,0.096,0.011,0.008,0.189,0.202
0.2,-0.005,-0.002,-0.018,0.206,0.010,0.010,0.154,0.326
0.3,-0.005,-0.003,0.112,0.102,0.011,0.011,0.148,0.402
0.5,0.001,-0.001,-0.147,0.034,0.010,0.008,0.110,0.425
1.0,0.018,0.008,-0.488,-0.341,0.011,0.013,0.372,0.439


In [3]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout

!git config user.name "Felix-prjcts"
!git config user.email "TON_ADRESSE_NOREPLY"     # voir plus bas
!git add cnn_fmnist.ipynb resultats/
!git commit -m "cnn fmnist : calibration + grille largeur 8, 20 epochs"

token = userdata.get("GITHUB_TOKEN")              # exactement le nom du secret dans le panneau 🔑
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git main 2>&1 | sed "s/{token}/***/g"

[Errno 2] No such file or directory: '/content/drive/MyDrive/early-dropout'
/content
fatal: not in a git directory
fatal: not in a git directory
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
GITHUB_TOKEN··········
fatal: not a git repository (or any of the parent directories): .git
